# Hindi ASR Production Dataset Generator

## Objective

Generate high-quality Hindi grocery and dairy store utterances using OpenRouter.

Pipeline:

Existing HuggingFace Dataset
        ↓
Conversation Generation
        ↓
JSON Validation
        ↓
Duplicate Removal
        ↓
Metadata Generation
        ↓
Append Dataset
        ↓
Auto Save
        ↓
Target: 5000 Records

In [21]:
# ==========================================
# IMPORTS
# ==========================================

import json
import time
import random
import re
import os

from datasets import load_dataset
from openai import OpenAI

print("=" * 60)
print("IMPORTS LOADED SUCCESSFULLY")
print("=" * 60)

IMPORTS LOADED SUCCESSFULLY


In [22]:
# ==========================================
# CONFIGURATION
# ==========================================

MODEL_NAME = "google/gemma-4-31b-it:free"

TARGET_RECORDS = 5000

TEMPERATURE = 0.9

MAX_TOKENS = 3000

BATCH_CONVERSATIONS = 5

SPEAKERS = [

    "kavya",

    "agastya",

    "maitri",

    "vinaya"

]

SCENARIOS = [

    "milk order with UPI payment",

    "paneer purchase",

    "curd and butter order",

    "festival grocery shopping",

    "home delivery request",

    "discount negotiation",

    "stock unavailable",

    "complaint about product",

    "bulk dairy purchase",

    "cash payment"

]

print("=" * 60)
print("CONFIGURATION LOADED")
print("=" * 60)

print()

print("Model :", MODEL_NAME)

print("Target :", TARGET_RECORDS)

CONFIGURATION LOADED

Model : google/gemma-4-31b-it:free
Target : 5000


In [23]:
# ==========================================
# OPENROUTER API KEY
# ==========================================

OPENROUTER_API_KEY = "PASTE_YOUR_OPENROUTER_API_KEY"

print("API KEY LOADED")

API KEY LOADED


In [24]:
# ==========================================
# OPENROUTER CLIENT
# ==========================================

client = OpenAI(

    base_url="https://openrouter.ai/api/v1",

    api_key=OPENROUTER_API_KEY

)

print("=" * 60)
print("OPENROUTER CLIENT READY")
print("=" * 60)

OPENROUTER CLIENT READY


In [25]:
# ==========================================
# LOAD HF DATASET
# ==========================================

dataset = load_dataset(

    "shujaAK/hindi-dairy-dataset-veena"

)

train_dataset = dataset["train"]

print("=" * 60)
print("DATASET LOADED")
print("=" * 60)

print()

print("Existing Records :", len(train_dataset))

DATASET LOADED

Existing Records : 850


In [26]:
# ==========================================
# PRODUCTION DATASET
# ==========================================

production_dataset = []

existing_transcripts = set()

for record in train_dataset:

    production_dataset.append({

        "transcript": record["transcript"],

        "domain_tag": record["domain_tag"],

        "language_mix": record["language_mix"],

        "sentence_length": record["sentence_length"],

        "speaker": record["speaker"],

        "deepgram_transcript": record["deepgram_transcript"],

        "deepgram_wer": record["deepgram_wer"],

        "deepgram_cer": record["deepgram_cer"]

    })

    existing_transcripts.add(

        record["transcript"].strip().lower()

    )

print("=" * 60)
print("PRODUCTION DATASET READY")
print("=" * 60)

print()

print("Current Records :", len(production_dataset))

PRODUCTION DATASET READY

Current Records : 850


In [27]:
# ==========================================
# GENERATE BATCH
# ==========================================

def generate_batch():

    scenario = random.choice(SCENARIOS)

    prompt = f"""
Generate exactly 5 realistic Indian grocery/dairy conversations.

Scenario:
{scenario}

Rules:

- Customer and shopkeeper only.
- Spoken Hindi.
- 8-12 utterances per conversation.
- Natural fillers.
- No repeated sentences.

Return ONLY valid JSON.

Format:

{{
    "conversations":[
        {{
            "conversation":[
                "...",
                "...",
                "..."
            ]
        }}
    ]
}}
"""

    response = client.chat.completions.create(

        model=MODEL_NAME,

        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],

        temperature=TEMPERATURE,

        max_tokens=MAX_TOKENS

    )

    return response.choices[0].message.content


print("="*60)
print("GENERATOR READY")
print("="*60)

GENERATOR READY


In [28]:
# ==========================================
# JSON PARSER
# ==========================================

def parse_openrouter_json(raw_output):

    start = raw_output.find("{")

    end = raw_output.rfind("}")

    json_text = raw_output[start:end+1]

    return json.loads(json_text)


print("="*60)
print("JSON PARSER READY")
print("="*60)

JSON PARSER READY


In [29]:
raw_output = generate_batch()

print(raw_output[:1500])

RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit exceeded: free-models-per-day. Add 10 credits to unlock 1000 free model requests per day', 'code': 429, 'metadata': {'headers': {'X-RateLimit-Limit': '50', 'X-RateLimit-Remaining': '0', 'X-RateLimit-Reset': '1782345600000'}, 'provider_name': None, 'previous_errors': [{'code': 429, 'message': 'Rate limit exceeded: free-models-per-day. Add 10 credits to unlock 1000 free model requests per day'}]}}, 'user_id': 'user_3FRzDtYDu1TYDWMKQ6lLYU7ejVS'}

In [ ]:
# ==========================================
# RUN ONE BATCH
# ==========================================

def run_one_batch():

    global production_dataset
    global existing_transcripts

    raw_output = generate_batch()

    parsed = parse_openrouter_json(raw_output)

    added = 0
    duplicates = 0

    for conversation in parsed["conversations"]:

        for sentence in conversation["conversation"]:

            transcript = sentence.strip()

            if transcript.lower() in existing_transcripts:

                duplicates += 1
                continue

            record = {

                "transcript": transcript,

                "domain_tag": "dairy_order",

                "language_mix": "hindi",

                "sentence_length": len(transcript.split()),

                "speaker": random.choice(SPEAKERS),

                "deepgram_transcript": "",

                "deepgram_wer": None,

                "deepgram_cer": None

            }

            production_dataset.append(record)

            existing_transcripts.add(transcript.lower())

            added += 1

    print("=" * 60)
    print("BATCH COMPLETED")
    print("=" * 60)

    print("Added :", added)
    print("Duplicates :", duplicates)
    print("Current Records :", len(production_dataset))


In [39]:
# ==========================================
# SAVE DATASET
# ==========================================

def save_dataset():

    os.makedirs("outputs", exist_ok=True)

    with open(
        "outputs/master_dataset.json",
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            production_dataset,
            f,
            ensure_ascii=False,
            indent=2
        )

    print("\n" + "="*60)
    print("CHECKPOINT SAVED SUCCESSFULLY")
    print("="*60)
    print("Total Records :", len(production_dataset))
    print("File :", "outputs/master_dataset.json")
    print("="*60)

In [ ]:
run_one_batch()

save_dataset()

In [41]:
# ============================================================
# CONTINUE TO 5000
# ============================================================

TARGET = 5000

while len(production_dataset) < TARGET:

    try:

        run_one_batch()

        current = len(production_dataset)

        if current % 250 < 60:
            save_dataset()

        print(f"\nProgress : {current}/{TARGET}")
        print(f"Remaining : {TARGET-current}")
        print("-"*60)

        time.sleep(3)

    except Exception as e:

        print("\nRetrying...")
        print(e)

        time.sleep(10)

save_dataset()

print("\n" + "="*60)
print("🎉 DATASET GENERATION COMPLETED")
print("="*60)
print("Final Records :", len(production_dataset))


Retrying...
Error code: 429 - {'error': {'message': 'Rate limit exceeded: free-models-per-day. Add 10 credits to unlock 1000 free model requests per day', 'code': 429, 'metadata': {'headers': {'X-RateLimit-Limit': '50', 'X-RateLimit-Remaining': '0', 'X-RateLimit-Reset': '1782345600000'}, 'provider_name': None, 'previous_errors': [{'code': 429, 'message': 'Rate limit exceeded: free-models-per-day. Add 10 credits to unlock 1000 free model requests per day'}]}}, 'user_id': 'user_3FRzDtYDu1TYDWMKQ6lLYU7ejVS'}

Retrying...
Error code: 429 - {'error': {'message': 'Rate limit exceeded: free-models-per-day. Add 10 credits to unlock 1000 free model requests per day', 'code': 429, 'metadata': {'headers': {'X-RateLimit-Limit': '50', 'X-RateLimit-Remaining': '0', 'X-RateLimit-Reset': '1782345600000'}, 'provider_name': None, 'previous_errors': [{'code': 429, 'message': 'Rate limit exceeded: free-models-per-day. Add 10 credits to unlock 1000 free model requests per day'}]}}, 'user_id': 'user_3FRzDt

KeyboardInterrupt: 

In [ ]:
print(len(production_dataset))

In [ ]:
print(os.listdir("outputs"))

In [ ]:
from google.colab import files

files.download("outputs/master_dataset.json")

In [32]:
with open("outputs/master_dataset.json","r",encoding="utf-8") as f:
    data=json.load(f)

print(len(data))

1762


In [33]:
print("Current Records :", len(production_dataset))

Current Records : 850


In [40]:
import json

with open(
    "master_dataset.json",
    "r",
    encoding="utf-8"
) as f:

    production_dataset = json.load(f)

print("="*60)
print("CHECKPOINT LOADED")
print("="*60)

print("Current Records :", len(production_dataset))

CHECKPOINT LOADED
Current Records : 1762


In [35]:
import os

print(os.getcwd())
print()
print(os.listdir("."))

/content

['.config', 'outputs', 'sample_data']


In [36]:
from google.colab import files

uploaded = files.upload()

Saving master_dataset.json to master_dataset.json


In [37]:
import os

print(os.listdir("."))

['.config', 'master_dataset.json', 'outputs', 'sample_data']


In [38]:
import json

with open("master_dataset.json", "r", encoding="utf-8") as f:
    production_dataset = json.load(f)

existing_transcripts = {
    item["transcript"].strip().lower()
    for item in production_dataset
}

print("=" * 60)
print("CHECKPOINT RESTORED")
print("=" * 60)
print("Current Records :", len(production_dataset))
print("Transcript Store:", len(existing_transcripts))

CHECKPOINT RESTORED
Current Records : 1762
Transcript Store: 1762
